# 03 · Qwen3-Embedding-4B: vectores e índice exacto

**TESIS-ISBD · QWEN3_4B_DENSO_RAG_V1**

## 1. Verificar catálogo, rutas e identidad

Abrir desde `07_RAG` o la raíz de `Tesis_Multimodelo`. La verificación es de los datos originales,
no una aprobación de las consultas. Esta celda no descarga ni ejecuta modelos.

In [1]:
import csv
import hashlib
import json
import sys
from pathlib import Path
from collections import Counter

VERSION = "QWEN3_4B_DENSO_RAG_V1"
MODELO = "Qwen/Qwen3-Embedding-4B"
REVISION = "5cf2132abc99cad020ac570b19d031efec650f2b"
LOTE = 4
DIMENSION_ESPERADA = 2560
LIMITE_DOCUMENTADO = 32768  # 32K documentados; no se truncarán resúmenes.
SHA_CATALOGO_ESPERADO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"


def sha(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


REPO = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
             if (p / "06_grafos" / "catalogo_articulos.csv").is_file()
             and (p / "07_RAG" / "auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta desde 07_RAG o Tesis_Multimodelo.")

BASE = REPO / "07_RAG"
RUTA_CATALOGO = REPO / "06_grafos" / "catalogo_articulos.csv"
RUTA_AUDITORIA = BASE / "auditoria_catalogo.json"
RUTA_CONSULTAS = BASE / "consultas_para_revision.csv"
VECTORES = BASE / "qwen3_4b_vectores_v1.npz"
INDICE = BASE / "qwen3_4b_indice_v1.faiss"

auditoria = json.loads(RUTA_AUDITORIA.read_text(encoding="utf-8"))
SHA_CATALOGO = sha(RUTA_CATALOGO)
if not (SHA_CATALOGO == SHA_CATALOGO_ESPERADO
        == auditoria["fuentes"]["catalogo"]["sha256_bytes"]):
    raise ValueError("El catálogo cambió respecto del utilizado con Qwen 0.6B y BGE-M3.")
SHA_AUDITORIA = sha(RUTA_AUDITORIA)

with RUTA_CATALOGO.open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))
article_ids = [r["article_id"] for r in articulos]
resumenes = [r["abstract"] for r in articulos]
if (len(article_ids) != 519 or len(set(article_ids)) != 519
        or len(set(resumenes)) != 514 or not all(t.strip() for t in resumenes)):
    raise ValueError("Estructura del catálogo distinta de la auditada.")

consultas = []
if RUTA_CONSULTAS.is_file():
    with RUTA_CONSULTAS.open(encoding="utf-8-sig", newline="") as f:
        consultas = list(csv.DictReader(f))
    print("Consultas:", len(consultas), "| Revisión humana:",
          dict(Counter(c.get("revision_humana", "no registrada") for c in consultas)))

print("Python:", sys.version.split()[0])
print("Catálogo verificado, SHA-256:", SHA_CATALOGO)
print("Artículos:", len(article_ids), "| Resúmenes distintos:", len(set(resumenes)))
print("Modelo:", MODELO, "| Revisión fijada:", REVISION)
print("Salidas:", VECTORES.name, "y", INDICE.name, "dentro de", BASE)
print("Esta celda NO ejecuta ningún modelo.")

Consultas: 24 | Revisión humana: {'pendiente': 24}
Python: 3.11.16
Catálogo verificado, SHA-256: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Artículos: 519 | Resúmenes distintos: 514
Modelo: Qwen/Qwen3-Embedding-4B | Revisión fijada: 5cf2132abc99cad020ac570b19d031efec650f2b
Salidas: qwen3_4b_vectores_v1.npz y qwen3_4b_indice_v1.faiss dentro de C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Esta celda NO ejecuta ningún modelo.


## 2. Comprobar bibliotecas y GPU

Se requieren las mismas bibliotecas que ya funcionaron. El perfil exige CUDA y soporte bfloat16; no se sustituye automáticamente por CPU o float32.

In [2]:
try:
    import numpy as np
    import torch
    import faiss
    import sentence_transformers
    import transformers
    import huggingface_hub
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Usa el mismo kernel donde funcionó BGE-M3; falta una biblioteca.") from exc

print("numpy:", np.__version__)
print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("faiss:", getattr(faiss, "__version__", "no declarada"))
if not torch.cuda.is_available():
    raise RuntimeError("Este perfil requiere CUDA. No se cambia a CPU automáticamente.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("No se detectó soporte bfloat16. Revisar antes de cambiar la precisión.")
print("GPU:", torch.cuda.get_device_name(0))
libre, total = torch.cuda.mem_get_info()
print("VRAM libre / total (GiB):", round(libre / 2**30, 2), "/", round(total / 2**30, 2))


def sincronizar():
    torch.cuda.synchronize()


def normalizar_l2_float32(salida, filas_esperadas, dimension_esperada):
    # La conversión de dtype no sustituye la normalización final.
    v = np.array(salida, dtype=np.float32, order="C", copy=True)
    if v.shape != (filas_esperadas, dimension_esperada) or not np.isfinite(v).all():
        raise ValueError("Forma o valores de embeddings incorrectos.")
    antes = np.linalg.norm(v, axis=1)
    if not np.isfinite(antes).all() or np.any(antes <= 0):
        raise ValueError("Vector nulo o norma no finita. No se guarda el índice.")
    faiss.normalize_L2(v)
    despues = np.linalg.norm(v, axis=1)
    if not np.isfinite(v).all() or not np.allclose(despues, 1, rtol=0, atol=1e-5):
        raise ValueError("Falló la normalización L2 en float32.")
    return v, antes, despues


def cargar_codificador():
    # La precisión se fija durante la carga, no después de cargar pesos en float32.
    modelo = SentenceTransformer(
        MODELO, revision=REVISION, device="cuda",
        model_kwargs={"torch_dtype": torch.bfloat16},
    )
    modelo.eval()
    if next(modelo.parameters()).dtype != torch.bfloat16:
        raise ValueError("La precisión cargada no coincide con bfloat16.")
    if modelo.get_sentence_embedding_dimension() != DIMENSION_ESPERADA:
        raise ValueError("Dimensión inesperada; no se recorta ni proyecta la salida.")
    if getattr(modelo, "default_prompt_name", None) is not None:
        raise ValueError("Existe un prefijo predeterminado inesperado para los documentos.")
    if not getattr(modelo, "prompts", {}).get("query"):
        raise ValueError("Falta el prefijo oficial query. No se inventa uno nuevo.")
    limite_original = int(modelo.max_seq_length)
    if limite_original <= 0:
        raise ValueError("Límite de entrada no válido.")
    # Evitar depender de un máximo interno distinto de los 32K documentados.
    modelo.max_seq_length = min(limite_original, LIMITE_DOCUMENTADO)
    return modelo, limite_original


print("Bibliotecas listas. Todavía no se ha cargado Qwen3-Embedding-4B.")

W1107 21:37:14.710000 4980 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


numpy: 2.3.5
torch: 2.11.0+cu128 | CUDA disponible: True
sentence-transformers: 6.2.0
transformers: 5.22.2
faiss: 1.15.1
GPU: NVIDIA GeForce RTX 4070 SUPER
VRAM libre / total (GiB): 10.81 / 11.99
Bibliotecas listas. Todavía no se ha cargado Qwen3-Embedding-4B.


In [3]:
from time import perf_counter
import torch

# Cargar el modelo sin generar todo el catálogo
inicio = perf_counter()
modelo_prueba, _ = cargar_codificador()
torch.cuda.synchronize()

print("Tiempo de carga:", round(perf_counter() - inicio, 2), "s")

# Probar únicamente cuatro resúmenes
inicio = perf_counter()

prueba = modelo_prueba.encode(
    resumenes[:4],
    batch_size=4,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
)

torch.cuda.synchronize()

print("Tiempo de cuatro resúmenes:", round(perf_counter() - inicio, 2), "s")
print("Dimensiones:", prueba.shape)
print("VRAM PyTorch reservada (GiB):",
      round(torch.cuda.memory_reserved() / 1024**3, 2))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

C:\Users\hazar\AppData\Local\Temp\ipykernel_4980\1007136659.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if modelo.get_sentence_embedding_dimension() != DIMENSION_ESPERADA:


Tiempo de carga: 9.63 s
Tiempo de cuatro resúmenes: 52.38 s
Dimensiones: (4, 2560)
VRAM PyTorch reservada (GiB): 7.65


In [4]:
from time import perf_counter
import torch

# Longitud en tokens de los cuatro resúmenes originales.
longitudes = [
    len(modelo_prueba.tokenizer(
        texto, add_special_tokens=True, truncation=False
    )["input_ids"])
    for texto in resumenes[:4]
]

print("Tokens de los cuatro resúmenes:", longitudes)

# Elegimos el más corto para una prueba individual.
i = longitudes.index(min(longitudes))

torch.cuda.synchronize()
inicio = perf_counter()

vector = modelo_prueba.encode(
    [resumenes[i]],
    batch_size=1,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

torch.cuda.synchronize()

print("Artículo:", article_ids[i])
print("Tokens:", longitudes[i])
print("Tiempo individual:", round(perf_counter() - inicio, 2), "s")
print("Dimensiones:", vector.shape)
print("VRAM reservada (GiB):",
      round(torch.cuda.memory_reserved() / 1024**3, 2))

Tokens de los cuatro resúmenes: [235, 163, 85, 181]
Artículo: 3
Tokens: 85
Tiempo individual: 0.93 s
Dimensiones: (1, 2560)
VRAM reservada (GiB): 7.66


In [5]:
from time import perf_counter
import torch

# Repetir exactamente los mismos cuatro resúmenes
# con el modelo ya cargado y utilizado.
torch.cuda.synchronize()
inicio = perf_counter()

vectores_prueba = modelo_prueba.encode(
    resumenes[:4],
    batch_size=4,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
)

torch.cuda.synchronize()
tiempo = perf_counter() - inicio

print("Tiempo del segundo lote:", round(tiempo, 2), "s")
print("Dimensiones:", vectores_prueba.shape)
print("VRAM reservada (GiB):",
      round(torch.cuda.memory_reserved() / 1024**3, 2))

Tiempo del segundo lote: 238.25 s
Dimensiones: (4, 2560)
VRAM reservada (GiB): 7.66


In [6]:
from time import perf_counter
import numpy as np
import torch

# Reutilizar el modelo ya cargado; no descargar ni cargar otra copia.
if "modelo_prueba" not in globals():
    raise RuntimeError(
        "El modelo de diagnóstico ya no está cargado. "
        "No ejecutes la vectorización completa."
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible en este kernel.")

parametro = next(modelo_prueba.parameters())
if parametro.device.type != "cuda" or parametro.dtype != torch.bfloat16:
    raise RuntimeError(
        f"Configuración distinta de la anterior: "
        f"{parametro.device}, {parametro.dtype}."
    )

# Contar tokens no genera embeddings ni modifica los textos.
tokenizados = modelo_prueba.tokenizer(
    resumenes,
    add_special_tokens=True,
    padding=False,
    truncation=False
)
longitudes = [len(ids) for ids in tokenizados["input_ids"]]
del tokenizados

# Los cuatro originales y el más largo, sin repetir identificadores.
mas_largo = max(range(len(resumenes)), key=lambda i: longitudes[i])
indices = list(dict.fromkeys([0, 1, 2, 3, mas_largo]))

# Límite operativo para no encadenar pruebas lentas.
# Se comprueba DESPUÉS de cada llamada; no es un timeout de CUDA.
LIMITE_POR_DOCUMENTO_S = 15.0
diagnostico_lote1 = []

print("Modelo:", parametro.device, "|", parametro.dtype)
print("Documentos que se comprobarán:", len(indices))
print("No se crearán archivos ni índices.\n")

for i in indices:
    if longitudes[i] > modelo_prueba.max_seq_length:
        raise ValueError(
            f"El artículo {article_ids[i]} excede el límite de entrada. "
            "No se truncará."
        )

    print(
        f"Procesando article_id={article_ids[i]} "
        f"| tokens={longitudes[i]}...",
        flush=True
    )

    torch.cuda.synchronize()
    inicio = perf_counter()

    salida = modelo_prueba.encode(
        [resumenes[i]],
        batch_size=1,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    torch.cuda.synchronize()
    segundos = perf_counter() - inicio

    salida = np.asarray(salida, dtype=np.float32)
    if salida.shape != (1, 2560) or not np.isfinite(salida).all():
        raise ValueError("La representación tiene forma o valores incorrectos.")
    if np.linalg.norm(salida[0]) == 0:
        raise ValueError("Se obtuvo un vector nulo.")

    diagnostico_lote1.append({
        "article_id": article_ids[i],
        "tokens": longitudes[i],
        "tiempo_s": round(segundos, 3)
    })
    print(f"Terminado: {segundos:.3f} s\n", flush=True)

    if segundos > LIMITE_POR_DOCUMENTO_S:
        print(
            "Se detiene antes del siguiente documento: "
            "esta llamada superó el límite operativo de 15 segundos."
        )
        break

print("\nRESULTADOS DEL DIAGNÓSTICO")
for resultado in diagnostico_lote1:
    print(resultado)

print(
    "Documentos comprobados:",
    len(diagnostico_lote1), "/", len(indices)
)

Modelo: cuda:0 | torch.bfloat16
Documentos que se comprobarán: 5
No se crearán archivos ni índices.

Procesando article_id=1 | tokens=235...
Terminado: 50.277 s

Se detiene antes del siguiente documento: esta llamada superó el límite operativo de 15 segundos.

RESULTADOS DEL DIAGNÓSTICO
{'article_id': '1', 'tokens': 235, 'tiempo_s': 50.277}
Documentos comprobados: 1 / 5


## 3. Generar los vectores y guardar el índice

Cambiar **solo `EJECUTAR = True`**. Se detiene si cualquiera de las dos salidas ya existe.

In [ ]:
EJECUTAR = True

if not EJECUTAR:
    print("Preparado. No se ha descargado Qwen3-Embedding-4B ni creado su índice.")
else:
    if VECTORES.exists() or INDICE.exists():
        raise FileExistsError("Ya existe una salida de Qwen 4B. No se sobrescribe; revisa antes de repetir.")
    if sha(RUTA_CATALOGO) != SHA_CATALOGO or sha(RUTA_AUDITORIA) != SHA_AUDITORIA:
        raise ValueError("Cambió una entrada desde la verificación. Ejecuta de nuevo la celda 1.")

    from time import perf_counter
    from datetime import datetime, timezone

    dispositivo = "cuda"
    fecha_inicio = datetime.now(timezone.utc).isoformat()
    libre_antes, total_vram = torch.cuda.mem_get_info()
    torch.cuda.reset_peak_memory_stats()
    sincronizar()
    inicio = perf_counter()
    codificador_qwen4b, limite_original = cargar_codificador()
    sincronizar()
    t_carga = perf_counter() - inicio
    dtype_modelo = str(next(codificador_qwen4b.parameters()).dtype)
    prefijo_consulta = codificador_qwen4b.prompts["query"]
    print("Modelo cargado en", dispositivo, "| Carga:", round(t_carga, 3), "s")
    print("Precisión del modelo:", dtype_modelo, "| Revisión:", REVISION)
    print("Prefijo de consulta oficial:", repr(prefijo_consulta))

    inicio = perf_counter()
    limite_tokens = int(codificador_qwen4b.max_seq_length)
    tokenizados = codificador_qwen4b.tokenizer(
        resumenes, add_special_tokens=True, padding=False, truncation=False
    )
    longitudes = np.asarray([len(x) for x in tokenizados["input_ids"]], dtype=np.int64)
    del tokenizados
    excedidos = [article_ids[i] for i in np.flatnonzero(longitudes > limite_tokens)]
    print("Tokens por resumen (mín./máx.):", int(longitudes.min()), int(longitudes.max()),
          "| Límite efectivo:", limite_tokens)
    if excedidos:
        raise ValueError("Resúmenes demasiado largos; no se truncaron: " + ", ".join(excedidos))
    t_longitudes = perf_counter() - inicio

    sincronizar()
    inicio = perf_counter()
    salida_encode = codificador_qwen4b.encode(
        resumenes, batch_size=LOTE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    sincronizar()
    t_vectorizacion = perf_counter() - inicio
    pico_asignado = int(torch.cuda.max_memory_allocated())
    pico_reservado = int(torch.cuda.max_memory_reserved())
    dtype_salida_encode = str(np.asarray(salida_encode).dtype)
    inicio = perf_counter()
    vectores, normas_antes, normas_despues = normalizar_l2_float32(
        salida_encode, len(article_ids), DIMENSION_ESPERADA
    )
    t_normalizacion = perf_counter() - inicio
    del salida_encode
    print("Salida de encode:", dtype_salida_encode)
    print("Normas ANTES (mín./máx.):", float(normas_antes.min()), float(normas_antes.max()))
    print("Normas DESPUÉS (mín./máx.):", float(normas_despues.min()), float(normas_despues.max()))
    print("Desviación máxima DESPUÉS:", float(np.max(np.abs(normas_despues - 1))))

    inicio = perf_counter()
    indice = faiss.IndexFlatIP(DIMENSION_ESPERADA)
    indice.add(vectores)
    t_indice = perf_counter() - inicio
    if indice.ntotal != len(article_ids):
        raise ValueError("El índice no contiene todos los artículos.")
    if sha(RUTA_CATALOGO) != SHA_CATALOGO:
        raise ValueError("El catálogo cambió durante la ejecución. No se guardan los resultados.")

    # Solo dos archivos de salida; no se guarda ninguna guía de pertinencia como entrada del modelo.
    with VECTORES.open("xb") as f:
        np.savez_compressed(
            f, vectores=vectores, article_ids=np.asarray(article_ids),
            modelo=np.asarray(MODELO), modelo_revision=np.asarray(REVISION),
            revision_origen=np.asarray("Commit explícito enviado a SentenceTransformer"),
            version_codigo=np.asarray(VERSION), modo=np.asarray("dense"),
            sha256_catalogo=np.asarray(SHA_CATALOGO), sha256_auditoria=np.asarray(SHA_AUDITORIA),
            fecha_inicio_utc=np.asarray(fecha_inicio),
            fecha_utc=np.asarray(datetime.now(timezone.utc).isoformat()),
            dispositivo=np.asarray(dispositivo), gpu_nombre=np.asarray(torch.cuda.get_device_name(0)),
            dtype_solicitado=np.asarray("bfloat16"), dtype_modelo=np.asarray(dtype_modelo),
            dtype_salida_encode=np.asarray(dtype_salida_encode),
            campo=np.asarray("abstract"), batch_size=np.asarray(LOTE),
            prefijo_documento=np.asarray(""), prefijo_consulta=np.asarray(prefijo_consulta),
            nombre_prompt_consulta=np.asarray("query"),
            max_seq_length_original=np.asarray(limite_original), max_seq_length=np.asarray(limite_tokens),
            tokens_por_articulo=longitudes, articulos_truncados=np.asarray(0),
            normalizacion_final=np.asarray("faiss.normalize_L2 sobre float32; rtol=0; atol=1e-5"),
            normas_antes_min=np.asarray(float(normas_antes.min())),
            normas_antes_max=np.asarray(float(normas_antes.max())),
            desviacion_norma_antes_max=np.asarray(float(np.max(np.abs(normas_antes - 1)))),
            normas_despues_min=np.asarray(float(normas_despues.min())),
            normas_despues_max=np.asarray(float(normas_despues.max())),
            desviacion_norma_despues_max=np.asarray(float(np.max(np.abs(normas_despues - 1)))),
            tiempo_carga_s=np.asarray(t_carga), tiempo_verificacion_longitud_s=np.asarray(t_longitudes),
            tiempo_vectorizacion_s=np.asarray(t_vectorizacion), tiempo_normalizacion_s=np.asarray(t_normalizacion),
            tiempo_construccion_indice_s=np.asarray(t_indice),
            gpu_libre_antes_bytes=np.asarray(int(libre_antes)), gpu_total_bytes=np.asarray(int(total_vram)),
            torch_cuda_max_memory_allocated_bytes=np.asarray(pico_asignado),
            torch_cuda_max_memory_reserved_bytes=np.asarray(pico_reservado),
            alcance_memoria=np.asarray("PyTorch en este proceso: carga y vectorización; no VRAM total ni RAM"),
            python=np.asarray(sys.version.split()[0]), numpy_version=np.asarray(np.__version__),
            torch_version=np.asarray(torch.__version__),
            sentence_transformers_version=np.asarray(sentence_transformers.__version__),
            transformers_version=np.asarray(transformers.__version__),
            huggingface_hub_version=np.asarray(huggingface_hub.__version__),
            faiss_version=np.asarray(getattr(faiss, "__version__", "no_registrada")),
        )
    if INDICE.exists():
        raise FileExistsError("El índice apareció durante la ejecución. No se reemplaza.")
    faiss.write_index(indice, str(INDICE))

    with np.load(VECTORES, allow_pickle=False) as guardado:
        if (guardado["article_ids"].tolist() != article_ids
                or guardado["sha256_catalogo"].item() != SHA_CATALOGO
                or not np.array_equal(guardado["vectores"], vectores)):
            raise ValueError("La relectura del NPZ no coincide con lo generado.")
    indice_guardado = faiss.read_index(str(INDICE))
    if (indice_guardado.ntotal != len(article_ids) or indice_guardado.d != DIMENSION_ESPERADA
            or not np.array_equal(indice_guardado.reconstruct_n(0, len(article_ids)), vectores)):
        raise ValueError("El índice guardado no coincide con los vectores.")

    print("Artículos indexados:", indice_guardado.ntotal, "| Dimensión:", indice_guardado.d,
          "| Vectorización:", round(t_vectorizacion, 3), "s")
    print("Pico PyTorch asignado / reservado (GiB):",
          round(pico_asignado / 2**30, 3), "/", round(pico_reservado / 2**30, 3))
    print("Guardados:", VECTORES.name, "y", INDICE.name)
    print("SHA-256 NPZ:", sha(VECTORES))
    print("SHA-256 FAISS:", sha(INDICE))
    print("No se han calculado métricas ni modificado resultados de los otros codificadores.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Modelo cargado en cuda | Carga: 9.479 s
Precisión del modelo: torch.bfloat16 | Revisión: 5cf2132abc99cad020ac570b19d031efec650f2b
Prefijo de consulta oficial: 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:'
Tokens por resumen (mín./máx.): 31 601 | Límite efectivo: 32768


C:\Users\hazar\AppData\Local\Temp\ipykernel_4980\1007136659.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if modelo.get_sentence_embedding_dimension() != DIMENSION_ESPERADA:


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

## 4. Búsqueda ilustrativa con el mismo codificador

La consulta se prepara con **`prompt_name="query"`**, como en Qwen 0.6B, y se normaliza
finalmente en float32. El prefijo procede del modelo fijado y se conserva literalmente en el NPZ.
No se añade el título, el identificador orientador o la respuesta esperada a la consulta.

Se utiliza la primera pregunta del CSV. Esta búsqueda no congela ni aprueba las 24 consultas.
Solo muestra cinco resultados; no informa precisión, recall ni nDCG. No crea otro archivo.

In [ ]:
if not VECTORES.is_file() or not INDICE.is_file():
    print("Primero completa la tercera celda con EJECUTAR=True.")
elif "codificador_qwen4b" not in globals():
    print("Los archivos están guardados. Para esta demostración carga solo el codificador, sin repetir el índice:")
    print("codificador_qwen4b, _ = cargar_codificador()")
else:
    with np.load(VECTORES, allow_pickle=False) as guardado:
        ids_guardados = guardado["article_ids"].tolist()
        prefijo_guardado = guardado["prefijo_consulta"].item()
        if (ids_guardados != article_ids or guardado["sha256_catalogo"].item() != SHA_CATALOGO
                or guardado["modelo"].item() != MODELO
                or guardado["modelo_revision"].item() != REVISION
                or prefijo_guardado != codificador_qwen4b.prompts.get("query")):
            raise ValueError("El NPZ no corresponde al catálogo, modelo o prefijo utilizado.")
        matriz_guardada = guardado["vectores"].copy()
    indice_busqueda = faiss.read_index(str(INDICE))
    if (indice_busqueda.ntotal != len(ids_guardados) or indice_busqueda.d != DIMENSION_ESPERADA
            or not np.array_equal(indice_busqueda.reconstruct_n(0, len(ids_guardados)), matriz_guardada)):
        raise ValueError("El índice no coincide con el NPZ.")
    del matriz_guardada

    q = (consultas[0]["consulta_propuesta"] if consultas else
         "¿Qué artículos estudian documentación técnica mediante RAG?")
    tokens_q = codificador_qwen4b.tokenizer(
        prefijo_guardado + q, add_special_tokens=True, truncation=False
    )["input_ids"]
    if len(tokens_q) > int(codificador_qwen4b.max_seq_length):
        raise ValueError("La consulta con su prefijo excede el límite; no se trunca.")
    salida_q = codificador_qwen4b.encode(
        [q], prompt_name="query", normalize_embeddings=True, convert_to_numpy=True
    )
    vq, _, _ = normalizar_l2_float32(salida_q, 1, indice_busqueda.d)
    _, posiciones = indice_busqueda.search(vq, min(5, indice_busqueda.ntotal))
    por_id = {r["article_id"]: r for r in articulos}
    print("Consulta de ejemplo:", q)
    print("Cinco resultados de Qwen3-Embedding-4B, SIN calificación de pertinencia:")
    for posicion in posiciones[0].tolist():
        if posicion >= 0:
            aid = ids_guardados[posicion]
            print(f"article_id={aid} | {por_id[aid]['titulo'][:95]}")